<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
👉 <b>Note:</b> This Colab notebook illustrates simplified usage of <code>rapidfireai</code>. For the full RapidFire AI experience with advanced experiment manager, UI, and production features, see our <a href=\"https://oss-docs.rapidfire.ai/en/latest/walkthrough.html\">Install and Get Started</a> guide.
<br/>
🎬 Watch our <a href=\"https://youtu.be/nPMBfZWqPWI\">intro video</a> to get started!\n
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RapidFireAI/rapidfireai/blob/main/tutorial_notebooks/fine-tuning/rf-colab-tensorboard-tutorial.ipynb)

⚠️ **IMPORTANT:** Do not let the Colab notebook tab stay idle for more than 5 minutes; Colab will disconnect otherwise. Refresh the TensorBoard screen or interact with the cells to avoid disconnection.

## Optimize Your Fine-tuning with RapidFire AI

This tutorial demonstrates how to fine-tune LLMs using **Supervised Fine-Tuning (SFT)** with [RapidFire AI](https://github.com/RapidFireAI/rapidfireai), enabling you to train and compare multiple configurations concurrently—even on a single GPU. We'll fine-tune a model on customer support data and use **TensorBoard** to monitor training metrics in real time.

## What We're Building

We'll fine-tune a **customer support chatbot** using [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), a modern and efficient 600M-parameter model (Apache 2.0 license). The model will learn to answer user queries in a helpful manner using the [Bitext Customer Support dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset), which contains instruction-response pairs covering common support scenarios.

Instead of training one configuration at a time and waiting hours for results, we'll use RapidFire AI to run **4 configurations simultaneously** and compare their training curves side-by-side in TensorBoard—all on a single GPU.

## Our Approach

We use **Supervised Fine-Tuning (SFT)** with **LoRA (Low-Rank Adaptation)** to efficiently adapt the pre-trained Qwen3-0.6B for customer support. To find the best hyperparameters, we compare **4 configurations** simultaneously:

- **2 LoRA adapter sizes**: Small (rank 8, attention Q/V only) vs. Large (rank 32, all attention projections)
- **2 learning rate schedules**: Linear decay (lr=5e-4) vs. Cosine decay with warmup (lr=2e-4)

RapidFire AI's **chunk-based scheduling** trains all configurations concurrently—splitting the dataset into chunks and letting every run train on each chunk before moving to the next. This provides comparative metrics early: after the first chunk completes, you can already see which configurations are performing better, instead of waiting for full training to finish. This approach delivers **16–24× faster experimentation throughput** compared to sequential training ([benchmark details](https://huggingface.co/blog/rapidfire-ai-inc/rapidfireai-sft-tutorial)).

### What You'll Learn

- Defining and running multiple SFT experiments concurrently with RapidFire AI
- Using LoRA adapters of different capacities for parameter-efficient fine-tuning
- Monitoring all training curves simultaneously in TensorBoard
- Using Interactive Control Operations (IC Ops) to Stop, Resume, Clone-Modify, and Delete runs mid-training

## Install RapidFire AI Package and Setup
### Option 1: Install Locally (or on a VM)
For the full RapidFire AI experience—advanced experiment management, UI, and production features—we recommend installing the package on a machine you control (for example, a VM or your local machine) rather than Google Colab. See our [Install and Get Started](https://oss-docs.rapidfire.ai/en/latest/walkthrough.html) guide.

### Option 2: Install in Google Colab
For simplicity, you can run this notebook on Google Colab. This notebook is configured to run end-to-end on Colab with no local installation required.

In [ ]:
try:
    import rapidfireai
    print("✅ rapidfireai already installed")
except ImportError:
    %pip install rapidfireai==0.15.1  # Takes 1 min
    !rapidfireai init # Takes 1 min

## Start RapidFire Services

RapidFire AI runs backend services including the **Dispatcher** (REST API on port 8851) and the **Frontend** dashboard (port 8853). The cell below checks whether these services are already running and launches them if not.

- If any issues arise, check the status in a terminal window using `rapidfireai status` or `rapidfireai doctor`.
- You can also run `rapidfireai start` from a terminal on Colab instead of the cell below.

In [ ]:
import subprocess
from time import sleep
import socket
try:
  s = [socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM)]
  s[0].connect(("127.0.0.1", 8851))
  s[1].connect(("127.0.0.1", 8852))
  s[2].connect(("127.0.0.1", 8853))
  s[0].close()
  s[1].close()
  s[2].close()
  print("RapidFire Services are running")
except OSError as error:
  print("RapidFire Services are not running, launching now...")
  subprocess.Popen(["rapidfireai", "start"])
  sleep(30)

## Configure TensorBoard

We use TensorBoard to visualize training loss, evaluation metrics, and learning rate schedules across all 4 runs simultaneously. This lets you compare configurations in real time as they train.

In [ ]:
import os

# Load TensorBoard extension
%load_ext tensorboard

# TensorBoard log directory will be auto-created in experiment path

## Import RapidFire Components

We import the core RapidFire classes:
- **`Experiment`**: Top-level container that groups runs, saves artifacts, and tracks metrics.
- **`RFGridSearch`**: Generates all combinations of your knobs into a Config Group.
- **`RFModelConfig`**: Wraps the base model, LoRA config, and training arguments into a single unit.
- **`RFLoraConfig` / `RFSFTConfig`**: RapidFire wrappers around PEFT's `LoraConfig` and TRL's `SFTConfig`, enabling grid-searchable parameters via `List()`.

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

# NB: If you get "AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'" from Colab, just rerun this cell

## Load Dataset

We use the [Bitext Customer Support dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset), which contains instruction-response pairs for common customer support scenarios (e.g., billing questions, account issues, product inquiries).

The dataset is reduced here to fit within Colab's memory constraints. For production training on a local machine, increase these ranges significantly (e.g., 10K+ samples).

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# REDUCED dataset for memory constraints in Colab
train_dataset = dataset["train"].select(range(64))  # Reduced from 128
eval_dataset = dataset["train"].select(range(50, 60))  # 10 examples
train_dataset = train_dataset.shuffle(seed=42)
eval_dataset = eval_dataset.shuffle(seed=42)

## Define Data Formatting Function

SFT training requires a consistent text format. We format each example as a simple Q&A pair that the model will learn to reproduce. The formatting function is passed to `RFModelConfig` and applied automatically during training.

In [ ]:
def sample_formatting_function(example):
    """Format the dataset for fine-tuning"""
    return {
        "text": f"Question: {example['instruction']}\nAnswer: {example['response']}"
    }

# Apply formatting to datasets
eval_dataset = eval_dataset.map(sample_formatting_function)
train_dataset = train_dataset.map(sample_formatting_function)

## Define Evaluation Metrics

We define a lightweight metrics function using ROUGE-L to measure the quality of generated responses against reference answers. ROUGE-L captures the longest common subsequence between prediction and reference, providing a reasonable proxy for response quality.

Only ROUGE-L is computed here (skipping BLEU) to conserve memory on Colab.

In [ ]:
def sample_compute_metrics(eval_preds):
    """Lightweight metrics computation"""
    predictions, labels = eval_preds

    try:
        import evaluate

        # Only compute ROUGE-L (skip BLEU to save memory)
        rouge = evaluate.load("rouge")
        rouge_output = rouge.compute(
            predictions=predictions,
            references=labels,
            use_stemmer=True,
            rouge_types=["rougeL"]  # Only compute rougeL
        )

        return {
            "rougeL": round(rouge_output["rougeL"], 4),
        }
    except Exception as e:
        # Fallback if metrics fail
        print(f"Metrics computation failed: {e}")
        return {}

## Initialize Experiment

Every RapidFire AI experiment needs a unique name. The `Experiment` object is the top-level container that groups all runs, saves artifacts, and tracks metrics. It automatically creates an MLflow experiment under the hood for structured logging.

In [ ]:
# Create experiment with unique name
my_experiment = "tensorboard-demo-1"
experiment = Experiment(experiment_name=my_experiment)

## Get TensorBoard Log Directory

RapidFire AI writes TensorBoard event files to a structured path inside the experiment directory. We retrieve this path so we can point TensorBoard at it for real-time monitoring.

In [ ]:
# Get experiment path
from rapidfireai.fit.db.rf_db import RfDb

db = RfDb()
experiment_path = db.get_experiments_path(my_experiment)
tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{my_experiment}"

print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

## Define Model Configurations

This is where RapidFire AI shines. Instead of hardcoding a single training configuration, we define a search space and let RapidFire run all combinations concurrently.

We use **Qwen3-0.6B** (600M parameters), a modern and efficient model with Apache 2.0 license. Two LoRA adapter configs are crossed with two learning rate schedules, producing **4 concurrent runs**.

### Quick LoRA Primer

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning technique that adds small trainable matrices to frozen model layers instead of updating all weights. This drastically reduces memory usage and training time. Key parameters:

- **`r` (rank)**: Dimensionality of the low-rank adapter matrices. Higher values = more capacity but more memory. We test rank 8 (lightweight) vs. rank 32 (higher capacity).
- **`lora_alpha`**: Scaling factor for LoRA weights, typically set to 2× the rank.
- **`target_modules`**: Which model layers to adapt. We compare adapting only Q/V attention projections vs. all four attention projections (Q/K/V/O).
- **`lora_dropout`**: Dropout rate applied to LoRA layers for regularization.

In [ ]:
# Qwen3 LoRA configs - standard transformer module names
peft_configs_lite = List([
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_proj", "v_proj"],  # Query and Value projections
        bias="none"
    ),
    RFLoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.1,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # All attention projections
        bias="none"
    )
])

# 2 configs with Qwen3-0.6B (modern, efficient model)
config_set_lite = List([
    RFModelConfig(
        model_name="Qwen/Qwen3-0.6B",  # 0.6B params, Apache 2.0 license
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=5e-4,  # Low lr for more stability
            lr_scheduler_type="linear",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,  # Effective bs = 4
            max_steps=64, # Raise this to see more learning
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            per_device_eval_batch_size=2,
            fp16=True,
            gradient_checkpointing=True,  # Save memory
            report_to="none",  # Disables wandb
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float16",
            "use_cache": False
        },
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 128,
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 40,
            "repetition_penalty": 1.1,
        }
    ),
    RFModelConfig(
        model_name="Qwen/Qwen3-0.6B",
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=2e-4,  # Even more conservative
            lr_scheduler_type="cosine",  # Try cosine schedule
            per_device_train_batch_size=2,
            gradient_accumulation_steps=2,
            max_steps=64, # Raise this to see more learning behaviors
            logging_steps=2,
            eval_strategy="steps",
            eval_steps=4,
            per_device_eval_batch_size=2,
            fp16=True,
            gradient_checkpointing=True,
            report_to="none",  # Disables wandb
            warmup_steps=10,  # Add warmup for stability
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float16",
            "use_cache": False
        },
        formatting_func=sample_formatting_function,
        compute_metrics=sample_compute_metrics,
        generation_config={
            "max_new_tokens": 128,
            "temperature": 0.7,
            "top_p": 0.9,
            "top_k": 40,
            "repetition_penalty": 1.1,
        }
    )
])

### Define Model Creation Function

RapidFire AI calls this function once per run to instantiate the model and tokenizer from the config dictionary. It receives the expanded config (with the specific LoRA/training args for that run) and must return a `(model, tokenizer)` tuple.

In [ ]:
def sample_create_model(model_config):
    """Function to create model object for any given config; must return tuple of (model, tokenizer)"""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_type = model_config["model_type"]
    model_kwargs = model_config["model_kwargs"]

    if model_type == "causal_lm":
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    else:
        raise ValueError(f"Unknown model_type: {model_type}. Supported types: 'causal_lm'")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Set pad token if not defined (common for decoder-only models)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id

    # Left padding is required for decoder-only causal LMs during batched generation
    tokenizer.padding_side = "left"

    return (model, tokenizer)

### Create Config Group

`RFGridSearch` takes the config set and generates a **Config Group**—the full Cartesian product of all `List()` knobs. Here, 2 LoRA configs × 2 training configs = **4 runs**. Each run in the Config Group will be trained concurrently via chunk-based scheduling.

In [ ]:
# Simple grid search across all config combinations: 4 total (2 LoRA configs × 2 trainer configs)
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type="SFT"
)

## Start TensorBoard

**IMPORTANT: Make sure to start TensorBoard BEFORE invoking run_fit() below so that you can watch metrics appear in real time!**

In [ ]:
%tensorboard --logdir {tensorboard_log_dir}

## Run Training + Validation

Now we launch `run_fit()`—the main function for running multi-config training and evaluation. RapidFire AI will:

1. **Expand configs** into 4 individual runs (one per LoRA × scheduler combination).
2. **Split the dataset** into `num_chunks=4` chunks for interleaved execution.
3. **Schedule all runs** on each chunk before moving to the next, so you get comparative metrics after the very first chunk.
4. **Log metrics** to TensorBoard in real time—scroll up to watch training loss and eval scores update live.

`num_chunks` controls the swap granularity: more chunks = more frequent comparisons but slightly more overhead from model swapping. 4–8 chunks work well for most experiments.

In [ ]:
# Launch train and validation for all configs in the config_group with swap granularity of 4 chunks for hyperparallel execution
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    eval_dataset,
    num_chunks=4,
    seed=42
)

## Launch Interactive Run Controller

RapidFire AI provides an Interactive Controller that lets you manage executing runs dynamically in real time from the notebook:

- ⏹️ **Stop**: Gracefully stop a running config
- ▶️ **Resume**: Resume a stopped run
- 🗑️ **Delete**: Remove a run from this experiment
- 📋 **Clone**: Create a new run by editing the config dictionary of a parent run to try new knob values; optional warm start of parameters
- 🔄 **Refresh**: Update run status and metrics

The Controller uses ipywidgets and is compatible with both Colab (ipywidgets 7.x) and Jupyter (ipywidgets 8.x).

In [ ]:
# Create Interactive Controller
sleep(15)
from rapidfireai.fit.utils.interactive_controller import InteractiveController

controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
controller.display()

## End Experiment

Always end the experiment to finalize logging, save all artifacts, and clean up resources. Click the button below when you're ready to wrap up.

In [ ]:
from google.colab import output
from IPython.display import display, HTML

display(HTML('''
<button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
'''))

# eval_js blocks until the Promise resolves
output.eval_js('''
new Promise((resolve) => {
    document.getElementById("continue-btn").onclick = () => {
        document.getElementById("continue-btn").disabled = true;
        document.getElementById("continue-btn").innerText = "Continuing...";
        resolve("clicked");
    };
})
''')

# Actually end the experiment after the button is clicked
experiment.end()
print("Done!")

## View TensorBoard Plots and Logs

After your experiment ends, you can still view the full logs in TensorBoard:

In [ ]:
# View final logs
%tensorboard --logdir {tensorboard_log_dir}

## View RapidFire AI Log Files

RapidFire AI produces structured log files for each experiment. These are useful for debugging, auditing training behavior, or understanding what the scheduler and workers did under the hood.

In [ ]:
# Get the experiment-specific log file
from IPython.display import display, Pretty
log_file = experiment.get_log_file_path()

display(Pretty(f"📄 Experiment Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# Get the training-specific log file
log_file = experiment.get_log_file_path("training")

display(Pretty(f"📄 Training Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

## Conclusion

In this tutorial, we fine-tuned a customer support chatbot by running **4 SFT configurations concurrently** on a single GPU using RapidFire AI's chunk-based scheduling. Here's what we covered:

1. **Defined a Config Group** using `List()` wrappers and `RFGridSearch` to specify multiple LoRA adapters and learning rate schedules.
2. **Ran all configurations concurrently** with `run_fit()`, getting comparative metrics after each data chunk instead of waiting for full training.
3. **Monitored training in real time** via TensorBoard, comparing loss curves and eval metrics side-by-side.
4. **Used IC Ops** to manage runs mid-training—stopping underperformers or cloning promising configurations with modified knobs.

### Next Steps

- **Try different models**: Replace Qwen3-0.6B with larger models like [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), [Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B), or [Phi-3-mini](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct).
- **Expand the grid**: Add more learning rates, LoRA ranks, or target module combinations to explore a wider hyperparameter space.
- **Scale up the dataset**: Increase from 64 training samples to 10K+ for production-quality fine-tuning on a local machine or VM.
- **Explore other training methods**: Use `RFDPOConfig` or `RFGRPOConfig` for preference alignment (DPO/GRPO) instead of SFT.
- **Use the RapidFire UI**: For a richer monitoring experience beyond TensorBoard, run RapidFire locally with the full dashboard at `http://localhost:8853`.

For more details, see the [full SFT blog post](https://huggingface.co/blog/rapidfire-ai-inc/rapidfireai-sft-tutorial) and the [RapidFire AI documentation](https://oss-docs.rapidfire.ai).